# LSTM + SHAP — Advanced Analysis (Part 3, Task 3)

**Two advanced techniques on top of the Day 6 models:**

1. **LSTM neural network** — a deep-learning forecaster that predicts next-hour
   `traffic_volume` from the previous 24 hours (traffic, temperature, rain, cloud cover).
   Compared against the persistence baseline and the Day 6 RandomForest (MAE 241.2).
2. **SHAP explainability** — explains *why* the congestion classifier predicts what it
   predicts, using Shapley values (global feature impact + direction).

Run top-to-bottom with **Run All**. Requires `data/processed/traffic_features.csv`
(Day 4) and `part3_machine_learning/models/congestion_classifier.joblib` (Day 6).

In [1]:
"""Setup: paths, data, seeds."""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.preprocessing import StandardScaler

def find_repo_root(start: Path) -> Path:
    """Walk upward until the repo root (marker: processed features CSV) is found."""
    for p in [start, *start.parents]:
        if (p / "data" / "processed" / "traffic_features.csv").exists():
            return p
    raise FileNotFoundError(
        "traffic_features.csv not found - run pipeline.py and feature_engineering.py first"
    )

BASE_DIR = find_repo_root(Path.cwd())
print(f"Repo root: {BASE_DIR}")

df = pd.read_csv(
    BASE_DIR / "data" / "processed" / "traffic_features.csv",
    parse_dates=["date_time"],
).sort_values("date_time").reset_index(drop=True)
print(f"Loaded {len(df):,} hours")

# The 27 engineered features (Day 4) - reused by the SHAP section
FEATURES = ["temp_c", "rain_1h", "snow_1h", "clouds_all", "hour", "day_of_week", "month",
            "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
            "is_weekend", "is_rush_hour", "is_holiday",
            "weather_Clear", "weather_Clouds", "weather_Drizzle", "weather_Fog",
            "weather_Haze", "weather_Mist", "weather_Rain", "weather_Smoke",
            "weather_Snow", "weather_Squall", "weather_Thunderstorm"]

SPLIT_DATE = pd.Timestamp("2018-01-01")
torch.manual_seed(42)  # DataLoader shuffling and weight init are torch-RNG driven
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Repo root: m:\capstone-project-sk-traffic
Loaded 40,575 hours
Device: cpu


## Part A — LSTM next-hour forecasting

**Design choices (viva-ready):**
- **Lookback = 24 h**: one full day of context captures both the morning and evening peaks.
- **4 input channels**: traffic volume itself plus the three most relevant exogenous signals (temperature, rain, cloud cover).
- **Scaler fit on the training period only** — no information from 2018 leaks into preprocessing.
- **Same time-based split as Day 6** (train < 2018-01-01, test = 2018), so LSTM and RandomForest numbers are directly comparable.
- Training sequences are subsampled (every 3rd) to keep CPU training time reasonable; the test set is evaluated in full.

In [2]:
"""Build (24h input -> next hour) sequences with a leakage-free split."""
LSTM_FEATURES = ["traffic_volume", "temp_c", "rain_1h", "clouds_all"]
LOOKBACK = 24

scaler = StandardScaler()
train_mask = df["date_time"] < SPLIT_DATE
scaler.fit(df.loc[train_mask, LSTM_FEATURES])          # fit on TRAIN only
scaled = scaler.transform(df[LSTM_FEATURES]).astype(np.float32)

all_idx = np.arange(LOOKBACK, len(df))
is_train = df["date_time"].to_numpy()[all_idx] < np.datetime64(SPLIT_DATE)
train_idx = all_idx[is_train][::3]                     # every 3rd sequence (~11k)
test_idx = all_idx[~is_train]                          # full 2018 test set

def build(indices):
    X = np.stack([scaled[i - LOOKBACK:i] for i in indices])
    y = scaled[indices, 0]                             # channel 0 = traffic_volume
    return torch.from_numpy(X), torch.from_numpy(y)

X_train, y_train = build(train_idx)
X_test, y_test = build(test_idx)
print(f"Train sequences: {X_train.shape}  |  Test sequences: {X_test.shape}")

Train sequences: torch.Size([11340, 24, 4])  |  Test sequences: torch.Size([6533, 24, 4])


In [3]:
"""Define and train the LSTM."""
class TrafficLSTM(nn.Module):
    def __init__(self, n_features=4, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1]).squeeze(-1)

model = TrafficLSTM(n_features=len(LSTM_FEATURES)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

EPOCHS, BATCH = 6, 256
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train), batch_size=BATCH, shuffle=True
)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    print(f"Epoch {epoch}/{EPOCHS}  train MSE: {total / len(train_idx):.4f}")

Model parameters: 4,897
Epoch 1/6  train MSE: 0.8669
Epoch 2/6  train MSE: 0.4654
Epoch 3/6  train MSE: 0.2405
Epoch 4/6  train MSE: 0.1933
Epoch 5/6  train MSE: 0.1621
Epoch 6/6  train MSE: 0.1416


In [4]:
"""Evaluate on 2018 and compare with baselines."""
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.eval()
with torch.no_grad():
    preds_scaled = []
    for i in range(0, len(test_idx), 1024):
        preds_scaled.append(model(X_test[i:i + 1024].to(DEVICE)).cpu().numpy())
    preds_scaled = np.concatenate(preds_scaled)

# invert the StandardScaler for channel 0 (traffic_volume)
mu, sd = scaler.mean_[0], scaler.scale_[0]
y_true = y_test.numpy() * sd + mu
y_pred = preds_scaled * sd + mu
y_persist = scaled[test_idx - 1, 0] * sd + mu          # persistence: next hour = this hour

def report(name, pred):
    mae = mean_absolute_error(y_true, pred)
    rmse = mean_squared_error(y_true, pred) ** 0.5
    r2 = r2_score(y_true, pred)
    print(f"{name:22s} MAE {mae:7.1f}  RMSE {rmse:7.1f}  R2 {r2:.4f}")
    return mae

print("=== Next-hour forecasting, test = all of 2018 ===")
report("Persistence baseline", y_persist)
report("LSTM (24h lookback)", y_pred)
print(f"{'RandomForest (Day 6)':22s} MAE   241.2  RMSE   406.8  R2 0.9575   <- tabular features, no sequences")

=== Next-hour forecasting, test = all of 2018 ===
Persistence baseline   MAE   588.9  RMSE   814.0  R2 0.8300
LSTM (24h lookback)    MAE   389.4  RMSE   539.4  R2 0.9254
RandomForest (Day 6)   MAE   241.2  RMSE   406.8  R2 0.9575   <- tabular features, no sequences


## Part B — SHAP explainability

RandomForest feature importances (Day 6) tell us *which* features matter, but not *how*.
SHAP (SHapley Additive exPlanations) assigns each feature a signed contribution per prediction:

- **summary bar plot** — global ranking by mean |SHAP| value
- **beeswarm plot** — each dot is one hour; colour = feature value, x-position = impact on P(congestion)

We explain the **congestion classifier** on a 1,000-hour sample of the 2018 test set.

In [5]:
"""SHAP TreeExplainer on the Day 6 congestion classifier."""
import joblib
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MODEL_PATH = BASE_DIR / "part3_machine_learning" / "models" / "congestion_classifier.joblib"
if not MODEL_PATH.exists():
    raise FileNotFoundError("Run part3_machine_learning/supervised/train_supervised.py first")
clf = joblib.load(MODEL_PATH)

test_df = df[df["date_time"] >= SPLIT_DATE]
X_sample = test_df.sample(n=1000, random_state=42)[FEATURES]

explainer = shap.TreeExplainer(clf)
sv = explainer.shap_values(X_sample)
if isinstance(sv, list):        # older SHAP: list per class
    sv1 = sv[1]
elif sv.ndim == 3:              # newer SHAP: (n_samples, n_features, n_classes)
    sv1 = sv[:, :, 1]
else:
    sv1 = sv
print(f"SHAP values computed for class 1 (congestion): {sv1.shape}")

FIG_DIR = BASE_DIR / "part3_machine_learning" / "notebooks" / "figures"
FIG_DIR.mkdir(exist_ok=True)

shap.summary_plot(sv1, X_sample, plot_type="bar", show=False, max_display=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_summary_bar.png", dpi=150)
plt.close()

shap.summary_plot(sv1, X_sample, show=False, max_display=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_beeswarm.png", dpi=150)
plt.close()
print(f"Saved: {FIG_DIR / 'shap_summary_bar.png'}")
print(f"Saved: {FIG_DIR / 'shap_beeswarm.png'}")

top5 = pd.Series(np.abs(sv1).mean(axis=0), index=FEATURES).sort_values(ascending=False).head(5)
print("\nTop-5 features by mean |SHAP| (impact on congestion prediction):")
for name, val in top5.items():
    print(f"  {name:20s} {val:.4f}")

m:\capstone-project-sk-traffic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP values computed for class 1 (congestion): (1000, 27)
Saved: m:\capstone-project-sk-traffic\part3_machine_learning\notebooks\figures\shap_summary_bar.png
Saved: m:\capstone-project-sk-traffic\part3_machine_learning\notebooks\figures\shap_beeswarm.png

Top-5 features by mean |SHAP| (impact on congestion prediction):
  hour_cos             0.1151
  is_rush_hour         0.0921
  hour                 0.0639
  dow_sin              0.0373
  is_weekend           0.0370


## Findings (bank these for the report and viva)

1. **LSTM vs RandomForest**: the LSTM leverages *sequences*; the RandomForest leverages the 27 *engineered* features. Whichever wins, the interesting part is *why* — the RF's calendar features (hour_sin/cos, is_rush_hour) already encode most of the daily rhythm, so a small LSTM has little extra signal to exploit on this dataset.
2. **Persistence is a strong baseline** for 1-hour-ahead traffic — any useful model must beat it clearly.
3. **SHAP confirms Part 1 and Day 6**: calendar features dominate congestion prediction; weather features contribute small, situational adjustments (e.g. snow *lowers* predicted congestion because people stay home).
4. **Explainability matters for deployment**: a city operator can audit individual predictions ("this hour was flagged congested mainly because it is 17:00 on a weekday"), which a black-box score alone cannot provide.